# 02 · Train Model A — the strict detector

The honest score. Trained **with** the adversarial rows, so it sees through
paraphrase-based humanization. DAMAGE (E8) reached 98.26% TPR on humanized AI
text at 5% FPR this way, against GPTZero's 60.04% and Binoculars' 28.23%.

**Loss: two-way partial AUROC, not cross-entropy** (E6, PRD 8.6). The costs
here are asymmetric — falsely accusing a human writer is materially worse than
missing a piece of AI text — so the objective optimises the low-FPR region of
the ROC curve, which is the only region A3 and A4 care about.

This model is **never** the humanizer's optimisation target (H5). Optimising
the rewrite against the detector that is meant to catch it is how a product
ends up grading its own homework.

Runtime: 4–8 hours on a T4. Checkpoints every 500 steps, so a session timeout
costs minutes, not the run.


In [ ]:
# Kaggle setup. Run once per session.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow

import sys, os
from pathlib import Path

# The repo is added as a Kaggle dataset, or cloned. Point REPO at it.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
print("repo:", REPO)


In [ ]:
import pandas as pd
from pathlib import Path
from lib.data import FEATURE_NAMES
from lib.train import TrainConfig, train

frame = pd.read_parquet(DATA / "train_a.parquet")

# Hold out by *generator and domain* where possible, not at random. A random
# split lets the model memorise a generator's quirks and score well on rows
# from the same generator, which is precisely the overfitting RAID exposed
# (E3: fine-tuned RoBERTa-Large averaged 56.7%).
holdout_domains = sorted(frame["domain"].unique())[-2:]
validation = frame[frame["domain"].isin(holdout_domains)]
training = frame[~frame["domain"].isin(holdout_domains)]
print(f"train {len(training):,} · validate {len(validation):,} on {holdout_domains}")

config = TrainConfig(
    backbone="microsoft/deberta-v3-base",
    max_length=768,
    batch_size=16,
    accumulation_steps=2,
    learning_rate=2e-5,
    epochs=3,
    fp16=True,
)

model, report = train(
    training, validation, list(FEATURE_NAMES),
    output_dir=WORK / "model_a",
    config=config,
)


In [ ]:
# Progress against the criteria this stage can measure (PRD 14).
final = report["final"]
print(f"AUROC            {final['auroc']:.4f}   (A1 needs >= 0.95 on the test split)")
print(f"partial AUROC@5% {final['partial_auroc_5']:.4f}")
print(f"TPR @ 1% FPR     {final['tpr_at_1_fpr']:.4f}   (A3 needs >= 0.80 on essays)")
print(f"TPR @ 5% FPR     {final['tpr_at_5_fpr']:.4f}   (A5 needs >= 0.90 on humanized AI)")
print("\nThese are validation-split numbers. The binding evaluation is notebook 05.")
